# Phase 2C Regime Comparison

This notebook is a curated, paper-facing analysis of the three Akash reputation regimes:

- Oracle (`[1.0, 1.0, 0.0]`)
- Headline (`[0.85, 0.85, 0.30]`)
- Degraded (`[0.70, 0.70, 0.60]`)

It reads local `run.json` artifacts and builds compact visualizations for:

1. Final utility-security summary (perplexity, ASR, timing)
2. Final degraded ASR-by-position
3. Degraded ASR trajectory across rounds

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN_FILES = {
    "oracle": ROOT / "results/akash_2c_oracle/run.json",
    "headline": ROOT / "results/akash_2c_headline_dseq26607956/run.json",
    "degraded": ROOT / "results/akash_2c_degraded_dseq26609636/run.json",
}

for name, path in RUN_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} artifact: {path}")


def load_run(path: Path) -> dict:
    return json.loads(path.read_text())


runs = {name: load_run(path) for name, path in RUN_FILES.items()}

In [ ]:
def final_round(run: dict) -> dict:
    return run["rounds"][-1]


summary_rows = []
for regime, run in runs.items():
    last = final_round(run)
    summary_rows.append(
        {
            "regime": regime,
            "perplexity": float(last["perplexity"]),
            "asr": float(last.get("asr", 0.0)),
            "round_time": float(last["round_time"]),
            "agg_time": float(last["agg_time"]),
            "asr_best_position": int(last.get("asr_best_position", 0)),
        }
    )

summary_rows

In [ ]:
labels = [row["regime"] for row in summary_rows]
perplexities = [row["perplexity"] for row in summary_rows]
asr_values = [row["asr"] for row in summary_rows]
agg_times = [row["agg_time"] for row in summary_rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(labels, perplexities, color=["#4C78A8", "#72B7B2", "#F58518"])
axes[0].set_title("Final Perplexity")
axes[0].set_ylabel("PPL")

axes[1].bar(labels, asr_values, color=["#4C78A8", "#72B7B2", "#F58518"])
axes[1].set_title("Final ASR")
axes[1].set_ylabel("ASR")

axes[2].bar(labels, agg_times, color=["#4C78A8", "#72B7B2", "#F58518"])
axes[2].set_title("Final Aggregation Time")
axes[2].set_ylabel("seconds")

for ax in axes:
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Phase 2C: Utility-Security-Timing Snapshot", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
degraded_last = final_round(runs["degraded"])
position_metrics = degraded_last["asr_per_position"]
positions = sorted(int(pos) for pos in position_metrics.keys())
position_asr = [float(position_metrics[str(pos)]) for pos in positions]

plt.figure(figsize=(8, 4))
plt.bar([str(p) for p in positions], position_asr, color="#E45756")
plt.title("Degraded Regime: Final ASR by Trigger Position")
plt.xlabel("Trigger Position")
plt.ylabel("ASR")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
degraded_rounds = runs["degraded"]["rounds"]
round_ids = [int(r["round"]) for r in degraded_rounds]
asr_trace = [float(r.get("asr", 0.0)) for r in degraded_rounds]
ppl_trace = [float(r["perplexity"]) for r in degraded_rounds]

fig, ax1 = plt.subplots(figsize=(10, 4))
line1 = ax1.plot(round_ids, asr_trace, color="#E45756", marker="o", label="ASR")
ax1.set_xlabel("Round")
ax1.set_ylabel("ASR", color="#E45756")
ax1.tick_params(axis="y", labelcolor="#E45756")
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
line2 = ax2.plot(round_ids, ppl_trace, color="#4C78A8", marker="s", label="Perplexity")
ax2.set_ylabel("Perplexity", color="#4C78A8")
ax2.tick_params(axis="y", labelcolor="#4C78A8")

lines = line1 + line2
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="upper right")

plt.title("Degraded Regime Trajectory: ASR vs Perplexity")
plt.tight_layout()
plt.show()

## Notes

- This notebook intentionally uses fixed artifact paths for reproducibility.
- For new deployments, duplicate this notebook and update `RUN_FILES` with new `dseq` directories.
- Prefer exporting publication-ready figures into `figures/phase2c/` via `plt.savefig(...)` in dedicated cells.